# Promote verified clean DEV tables to PROD

Manual maintenance only. Finish DEV `monthly`, `daily`, `validate` and obtain PASS first.
Run one stage at a time: `plan` → `copy` → `validate`. Do not use Run all.
PLAN reads Databricks and saves a private checkpoint. COPY replaces the 30 existing
managed PROD business tables using DEEP CLONE at frozen DEV Delta versions. No PROD
reset or second ingestion/backfill is needed. RAW/GCS and DEV are not written.

OPS tables and entitlements are not cloned. Real promotion events are added to OPS;
validation updates PROD month states and only the approved application display label.
Ingestion IDs retain their DEV provenance. No fake monthly/daily PROD runs are created.
A partial copy is NOT an atomic publication: keep jobs and dashboard paused throughout.
On failure preserve checkpoints, stop and diagnose; no automatic retry or RESTORE.
Read the promotion section in `docs/privacy_rebuild.md` before execution.


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'src/finops_cloud').is_dir()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('/Workspace/Users/thailongdam@gmail.com/finops-cloud_data_platform')
if not (PROJECT_ROOT / 'src/finops_cloud').is_dir():
    raise ValueError('Set PROJECT_ROOT to the Belgian Databricks Git Folder')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from finops_cloud.config import load_config
from finops_cloud.medallion.promotion import run_stage


## Parameters

Use the existing private Workspace folder outside Git. Keep `rebuild_dev.json`;
this notebook creates a separate `promote_dev_to_prod.json`, never a new DEV baseline.
Leave confirmations empty/False for PLAN. For COPY/VALIDATE, set the exact confirmation
`PROMOTE_CLEAN_DEV_TO_PROD` only after suspending jobs and actual public dashboard reads.


In [ ]:
STAGE = 'plan'
CONTROL_DIRECTORY = Path('/Workspace/Users/thailongdam@gmail.com/privacy-rebuild-20261006')
CONFIRMATION = ''
JOBS_PAUSED = False
DASHBOARD_PAUSED = False

dev_config = load_config('dev', PROJECT_ROOT)
prod_config = load_config('prod', PROJECT_ROOT)
dev_checkpoint = CONTROL_DIRECTORY / 'rebuild_dev.json'
state_file = CONTROL_DIRECTORY / 'promote_dev_to_prod.json'
print('Review:', {'stage': STAGE, 'source': dev_config.catalog,
                  'target': prod_config.catalog, 'checkpoint': str(state_file)})


In [ ]:
result = run_stage(
    spark, dev_config, prod_config, state_file, dev_checkpoint,
    stage=STAGE, confirmation=CONFIRMATION,
    jobs_paused=JOBS_PAUSED, dashboard_paused=DASHBOARD_PAUSED,
)
print(json.dumps(result, indent=2))
if STAGE == 'plan':
    control = json.loads(state_file.read_text(encoding='utf-8'))
    print('PROD recovery versions — save privately:', json.dumps(control['recovery_versions']))
    print('Pinned DEV versions — save privately:', json.dumps(control['source_versions']))


## After PROD PASS

Validation reuses the rebuild controls and executes existing serving scripts 04/05.
It does not prove end-to-end authorization. Test Owner A, Owner B, Admin and No Access,
including exports and periods, before publishing or resuming jobs/dashboard reads.
Keep historical data, OPS logs, grant snapshots and recovery checkpoints private.
DEEP CLONE avoids rerunning transformations but still uses compute and storage.
